In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
import h5py
import numpy as np
import os
from scipy.ndimage import label

# === CONFIG ===
input_dir = "/content/gdrive/MyDrive/tcc/ISRO TCC"  # Directory containing HDF5 files
output_input_dir = "./cnn_inputs"  # Directory for CNN input arrays
output_label_dir = "./cnn_labels"  # Directory for CNN label arrays

os.makedirs(output_input_dir, exist_ok=True)
os.makedirs(output_label_dir, exist_ok=True)

# === Helper: Calculate cluster properties ===
def compute_cluster_stats(region_pixels, TIR1_temperature, X, Y):
    indices = np.argwhere(region_pixels)
    tb_values = TIR1_temperature[region_pixels]
    min_tb = tb_values.min()
    mean_tb = tb_values.mean()
    median_tb = np.median(tb_values)
    std_tb = tb_values.std()

    y_center, x_center = indices[tb_values.argmin()]  # coldest point
    lon_center = X[x_center]
    lat_center = Y[y_center]

    dx_km = np.abs(X[1] - X[0]) * 111
    dy_km = np.abs(Y[1] - Y[0]) * 111
    pixel_area_km2 = dx_km * dy_km
    area_km2 = np.sum(region_pixels) * pixel_area_km2
    radius_km = np.sqrt(area_km2 / np.pi)

    return {
        'lat_center': lat_center,
        'lon_center': lon_center,
        'pixel_count': np.sum(region_pixels),
        'mean_tb': mean_tb,
        'min_tb': min_tb,
        'median_tb': median_tb,
        'std_tb': std_tb,
        'radius_km': radius_km,
        'area_km2': area_km2
    }

# === PROCESS FILES ===
for filename in os.listdir(input_dir):
    if filename.endswith(".h5"):
        file_path = os.path.join(input_dir, filename)
        with h5py.File(file_path, 'r') as f:
            IMG_TIR1 = f['IMG_TIR1'][0]
            IMG_TIR1_TEMP = f['IMG_TIR1_TEMP'][:]
            X = f['X'][:]
            Y = f['Y'][:]

        IMG_TIR1 = np.clip(IMG_TIR1, 0, IMG_TIR1_TEMP.shape[0] - 1)
        TIR1_temperature = IMG_TIR1_TEMP[IMG_TIR1]
        TCC_mask = TIR1_temperature < 235

        structure = np.ones((3,3))
        labeled_mask, num_features = label(TCC_mask, structure=structure)

        dx_km = np.abs(X[1] - X[0]) * 111
        dy_km = np.abs(Y[1] - Y[0]) * 111
        pixel_area_km2 = dx_km * dy_km
        min_radius_km = 0.9 * 111
        min_area_km2 = np.pi * (min_radius_km**2)

        final_TCC_mask = np.zeros_like(TCC_mask)
        cluster_stats = []

        for region_id in range(1, num_features + 1):
            region_pixels = (labeled_mask == region_id)
            area_km2 = np.sum(region_pixels) * pixel_area_km2
            radius_km = np.sqrt(area_km2 / np.pi)
            if radius_km >= min_radius_km:
                final_TCC_mask |= region_pixels
                stats = compute_cluster_stats(region_pixels, TIR1_temperature, X, Y)
                cluster_stats.append(stats)

        # === SAVE CNN INPUT & LABEL ===
        input_out = os.path.join(output_input_dir, filename.replace(".h5", "_input.npy"))
        label_out = os.path.join(output_label_dir, filename.replace(".h5", "_label.npy"))

        np.save(input_out, TIR1_temperature)
        np.save(label_out, final_TCC_mask.astype(np.uint8))

        # Optionally: Save stats as CSV or JSON
        # Example: json_out = os.path.join(output_label_dir, filename.replace(".h5", "_stats.json"))
        # with open(json_out, 'w') as jf:
        #     json.dump(cluster_stats, jf, indent=2)

        print(f"Processed {filename}: {len(cluster_stats)} TCCs found.")

print("✅ All files processed with TCC stats ready for CNN training.")

Processed 3DIMG_01JUL2023_0000_L1C_ASIA_MER_V01R00.h5: 1025 TCCs found.


In [ ]:
import shutil

# Replace 'my_folder' with your folder name
shutil.make_archive('my_folder', 'zip', 'my_folder')


In [ ]:
from google.colab import files
files.download('my_folder.zip')


In [ ]:
"""
unet_effnet_tf.py
U-Net style segmentation with a pretrained EfficientNetB0 encoder (transfer learning)
TensorFlow / Keras implementation.

- Binary segmentation by default (n_classes=1 -> sigmoid). Change n_classes>1 for multiclass.
- Uses tf.data for performance.
- Optional encoder freezing (freeze_backbone=True).
- BCE + Dice combination loss included.
- Data augmentation using tf.image (no external libs required).
"""

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import os

# ---------------------------
# Basic config (edit as needed)
# ---------------------------
IMG_SIZE = (256, 256)
BATCH_SIZE = 8
AUTOTUNE = tf.data.experimental.AUTOTUNE
BUFFER_SIZE = 512
SEED = 42

# ---------------------------
# Losses & metrics
# ---------------------------
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    """
    y_true, y_pred: shape [B, H, W, C]
    Works for binary (C=1) and multiclass (one-hot).
    """
    y_true_f = tf.reshape(y_true, (tf.shape(y_true)[0], -1))
    y_pred_f = tf.reshape(y_pred, (tf.shape(y_pred)[0], -1))
    intersection = tf.reduce_sum(y_true_f * y_pred_f, axis=1)
    denom = tf.reduce_sum(y_true_f + y_pred_f, axis=1)
    dice = (2. * intersection + smooth) / (denom + smooth)
    return tf.reduce_mean(dice)

class DiceLoss(tf.keras.losses.Loss):
    def call(self, y_true, y_pred):
        return 1.0 - dice_coefficient(y_true, y_pred)

class BCEWithDiceLoss(tf.keras.losses.Loss):
    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce_weight = bce_weight
        self.bce = tf.keras.losses.BinaryCrossentropy(from_logits=False)  # expect probabilities as model output for binary
        self.dice = DiceLoss()

    def call(self, y_true, y_pred):
        # y_pred expected in probability space (sigmoid applied in model for binary)
        bce_loss = self.bce(y_true, y_pred)
        dice_loss = self.dice(y_true, y_pred)
        return self.bce_weight * bce_loss + (1 - self.bce_weight) * dice_loss

# For multiclass you may use categorical crossentropy + generalized dice.

# ---------------------------
# Small decoder block
# ---------------------------
def decoder_block(x, skip, filters, up_size=(2,2), name=None):
    x = layers.UpSampling2D(size=up_size, interpolation='bilinear')(x)
    if skip is not None:
        # if shapes mismatch due to odd dims, resize
        if x.shape[1] != skip.shape[1] or x.shape[2] != skip.shape[2]:
            x = tf.image.resize(x, (skip.shape[1], skip.shape[2]), method='bilinear')
        x = layers.Concatenate()([x, skip])
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x

# ---------------------------
# Build model: EfficientNetB0 encoder + U-Net decoder
# ---------------------------
def build_unet_effnet(input_shape=(256,256,3), n_classes=1, pretrained=True, freeze_backbone=False):
    """
    n_classes=1 -> binary segmentation with sigmoid output (shape HxW x 1)
    n_classes>1 -> multiclass segmentation with softmax output (shape HxW x n_classes)
    """
    inputs = layers.Input(shape=input_shape)

    # Preprocessing layer compatible with EfficientNet
    preproc = tf.keras.applications.efficientnet.preprocess_input(inputs)

    # Load backbone
    base = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights='imagenet' if pretrained else None,
        input_tensor=preproc
    )

    if freeze_backbone:
        base.trainable = False
    else:
        base.trainable = True

    # Encoder feature maps for skip connections (choose layers that give multiple scales)
    # Names observed from EfficientNetB0: 'block2a_expand_activation', 'block3a_expand_activation', 'block4a_expand_activation', 'block6a_expand_activation'
    skip_names = [
        "block2a_expand_activation",  # 64x64 (for 256 input)
        "block3a_expand_activation",  # 32x32
        "block4a_expand_activation",  # 16x16
        "block6a_expand_activation",  # 8x8
    ]
    skips = [base.get_layer(name).output for name in skip_names]
    x = base.output  # bottleneck (e.g., 8x8 -> 512 channels)

    # Center convs
    x = layers.Conv2D(512, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Decoder (reverse order of skips)
    x = decoder_block(x, skips[-1], 256)  # combine with block6a
    x = decoder_block(x, skips[-2], 128)  # combine with block4a
    x = decoder_block(x, skips[-3], 64)   # combine with block3a
    x = decoder_block(x, skips[-4], 32)   # combine with block2a

    # Final upsample to original resolution if necessary
    x = layers.UpSampling2D(size=(2,2), interpolation='bilinear')(x)  # bring to input resolution
    x = layers.Conv2D(32, 3, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    if n_classes == 1:
        outputs = layers.Conv2D(1, 1, activation='sigmoid')(x)
    else:
        outputs = layers.Conv2D(n_classes, 1, activation='softmax')(x)

    model = keras.Model(inputs, outputs)
    return model

# ---------------------------
# Data pipeline helpers
# ---------------------------
def read_image(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3)
    img.set_shape([None, None, 3])
    img = tf.image.convert_image_dtype(img, tf.float32)  # [0,1]
    return img

def read_mask(path, n_classes=1):
    m = tf.io.read_file(path)
    m = tf.image.decode_image(m, channels=1)
    m.set_shape([None, None, 1])
    m = tf.image.convert_image_dtype(m, tf.float32)  # [0,1] float
    if n_classes == 1:
        # ensure binary mask (0 or 1)
        m = tf.where(m > 0.5, 1.0, 0.0)
    else:
        # assume mask is integer-coded classes (H,W,1) with values [0..n_classes-1]
        # convert to one-hot
        m = tf.cast(tf.squeeze(m, axis=-1), tf.int32)
        m = tf.one_hot(m, n_classes)
    return m

def preprocess_pair(img_path, mask_path, img_size=IMG_SIZE, n_classes=1, augment=False):
    img = read_image(img_path)
    mask = read_mask(mask_path, n_classes)

    # Resize both
    img = tf.image.resize(img, img_size)
    mask = tf.image.resize(mask, img_size, method='nearest')

    if augment:
        # Random flip
        if tf.random.uniform(()) > 0.5:
            img = tf.image.flip_left_right(img)
            mask = tf.image.flip_left_right(mask)
        # Random brightness (small)
        img = tf.image.random_brightness(img, 0.1)
        # Random contrast
        img = tf.image.random_contrast(img, 0.9, 1.1)
        # Random rotation by 0,90,180,270 (cheap)
        k = tf.random.uniform((), minval=0, maxval=4, dtype=tf.int32)
        img = tf.image.rot90(img, k)
        mask = tf.image.rot90(mask, k)

    # Optionally: apply standard ImageNet normalization - handled by backbone preprocess_input in model
    return img, mask

def make_dataset(image_paths, mask_paths, batch_size=BATCH_SIZE, img_size=IMG_SIZE, n_classes=1, shuffle=True, augment=False):
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
    if shuffle:
        dataset = dataset.shuffle(BUFFER_SIZE, seed=SEED)
    dataset = dataset.map(lambda x,y: preprocess_pair(x,y,img_size,n_classes,augment), num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(batch_size).prefetch(AUTOTUNE)
    return dataset

# ---------------------------
# Example usage / training
# ---------------------------
def compile_and_train(train_images, train_masks, val_images, val_masks,
                      n_classes=1,
                      out_dir="models",
                      epochs=30,
                      lr=1e-4,
                      freeze_backbone=False,
                      mixed_precision=False):
    if mixed_precision:
        # enable mixed precision
        tf.keras.mixed_precision.set_global_policy('mixed_float16')
        print("Mixed precision enabled")

    model = build_unet_effnet(input_shape=(*IMG_SIZE, 3), n_classes=n_classes, pretrained=True, freeze_backbone=freeze_backbone)
    model.summary()

    # Loss & metrics
    if n_classes == 1:
        loss = BCEWithDiceLoss(bce_weight=0.5)
        metrics = [tf.keras.metrics.MeanIoU(num_classes=2), keras.metrics.AUC()]  # IoU expects integer classes - for binary it will work but be careful
    else:
        loss = tf.keras.losses.CategoricalCrossentropy()
        metrics = [tf.keras.metrics.MeanIoU(num_classes=n_classes)]

    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    model.compile(optimizer=optimizer, loss=loss, metrics=[dice_coefficient] + metrics)

    # Prepare datasets
    train_ds = make_dataset(train_images, train_masks, batch_size=BATCH_SIZE, img_size=IMG_SIZE, n_classes=n_classes, shuffle=True, augment=True)
    val_ds = make_dataset(val_images, val_masks, batch_size=BATCH_SIZE, img_size=IMG_SIZE, n_classes=n_classes, shuffle=False, augment=False)

    # Callbacks
    os.makedirs(out_dir, exist_ok=True)
    checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(os.path.join(out_dir, 'best_model.h5'),
                                                       save_best_only=True, monitor='val_loss')
    reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
    early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
    csv_logger = tf.keras.callbacks.CSVLogger(os.path.join(out_dir, 'training_log.csv'))

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=[checkpoint_cb, reduce_lr, early_stop, csv_logger]
    )
    return model, history

# ---------------------------
# Inference helper
# ---------------------------
def predict_mask(model, image_tensor, threshold=0.5):
    """
    image_tensor: HxWx3 float32 in [0,1]
    returns: HxW binary mask (numpy) for binary segmentation.
    """
    img = tf.image.resize(image_tensor, IMG_SIZE)
    img = tf.expand_dims(img, axis=0)
    preds = model.predict(img)  # for binary -> shape (1,H,W,1) probabilities
    pred = preds[0]
    if pred.shape[-1] == 1:
        mask = (pred[...,0] > threshold).astype(np.uint8)
    else:
        mask = np.argmax(pred, axis=-1).astype(np.uint8)
    # resize back to original shape if desired (use cv2 or tf.image.resize)
    return mask

# ---------------------------
# Example quick-run (replace with your paths)
# ---------------------------
if __name__ == "__main__":
    # Example: point these to your file lists (strings)
    # Make sure lists are sorted/paired correctly (image i -> mask i)
    train_images = sorted([f"/path/to/train/images/{p}" for p in os.listdir("/path/to/train/images")])
    train_masks = sorted([f"/path/to/train/masks/{p}" for p in os.listdir("/path/to/train/masks")])
    val_images = sorted([f"/path/to/val/images/{p}" for p in os.listdir("/path/to/val/images")])
    val_masks = sorted([f"/path/to/val/masks/{p}" for p in os.listdir("/path/to/val/masks")])

    # Quick sanity
    assert len(train_images) == len(train_masks), "Train images and masks count mismatch"
    assert len(val_images) == len(val_masks), "Val images and masks count mismatch"

    model, history = compile_and_train(train_images, train_masks, val_images, val_masks,
                                       n_classes=1,
                                       out_dir="models",
                                       epochs=50,
                                       lr=1e-4,
                                       freeze_backbone=False,
                                       mixed_precision=False)

    # Example inference:
    # from PIL import Image
    # img = np.array(Image.open(train_images[0]).convert('RGB'))/255.0
    # mask_pred = predict_mask(model, img)
    # print(mask_pred.shape)

\

In [ ]:
#     datset genrator for my lstm model for tracking purpose

In [ ]:
import numpy as np
import h5py, os
from scipy.ndimage import label
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
import pandas as pd

# -------------------------
# Geo helpers
# -------------------------
def haversine_km(latlon_a, latlon_b):
    """latlon_a: (N,2) [lat,lon], latlon_b: (M,2)"""
    # Convert to radians
    a = np.radians(latlon_a)
    b = np.radians(latlon_b)
    d = b[:,None,:] - a[None,:,:]   # (M,N,2)
    dlat = d[:,:,0]; dlon = d[:,:,1]
    lat1 = a[:,0]; lat2 = b[:,0][:,None]
    R = 6371.0
    h = np.sin(dlat/2)*2 + np.cos(lat1)[None,:]*np.cos(lat2)*np.sin(dlon/2)*2
    return 2*R*np.arcsin(np.sqrt(h))  # (M,N) distances MxN (curr x prev)

# -------------------------
# Feature extraction
# -------------------------
def compute_cluster_features(region_mask, T, X, Y):
    ys, xs = np.where(region_mask)
    # centroid as mean index
    y_c = int(np.round(ys.mean())); x_c = int(np.round(xs.mean()))
    lat = float(Y[y_c]); lon = float(X[x_c])

    tb_vals = T[ys, xs]
    min_tb = float(tb_vals.min()); mean_tb = float(tb_vals.mean())

    # pixel size (approx) -> km^2
    dy_km = float(abs(Y[1]-Y[0]) * 111.0)
    dx_km = float(abs(X[1]-X[0]) * 111.0)
    pixel_area_km2 = dx_km * dy_km
    area_km2 = float(len(xs) * pixel_area_km2)

    bbox = (int(ys.min()), int(xs.min()), int(ys.max()), int(xs.max()))
    return {
        "lat": lat, "lon": lon,
        "area_km2": area_km2,
        "min_tb": min_tb, "mean_tb": mean_tb,
        "bbox": bbox,
        "pix_count": int(len(xs))
    }

def extract_clusters_from_mask(mask_bool, T, X, Y, min_radius_km=100.0):
    """Returns list of cluster dicts with per-cluster features."""
    structure = np.ones((3,3))
    labeled, n = label(mask_bool, structure=structure)

    clusters = []
    # pixel area to radius filter
    dy_km = float(abs(Y[1]-Y[0]) * 111.0)
    dx_km = float(abs(X[1]-X[0]) * 111.0)
    pixel_area_km2 = dx_km * dy_km
    min_area_km2 = np.pi * (min_radius_km**2)

    for rid in range(1, n+1):
        region = (labeled == rid)
        area_km2 = region.sum() * pixel_area_km2
        if area_km2 < min_area_km2:
            continue
        feat = compute_cluster_features(region, T, X, Y)
        feat["region_id"] = rid
        clusters.append(feat)
    return clusters

# -------------------------
# Cost & matching
# -------------------------
def build_cost(prev_list, curr_list, w_dist=1.0, w_area=0.3, w_temp=0.1, max_dist_km=500):
    """Return cost matrix (prev x curr) and a mask of invalid pairs (for gating)."""
    if len(prev_list)==0 or len(curr_list)==0:
        return np.zeros((len(prev_list), len(curr_list))), np.ones((len(prev_list), len(curr_list)), dtype=bool)

    prev_xy = np.array([[p["lat"], p["lon"]] for p in prev_list])
    curr_xy = np.array([[c["lat"], c["lon"]] for c in curr_list])

    # distances: matrix (curr x prev); we'll transpose to (prev x curr)
    D = haversine_km(prev_xy, curr_xy).T  # (curr x prev) -> transpose later
    # Normalize-ish: keep in km, but we gate later
    # area change (relative)
    A_prev = np.array([p["area_km2"] for p in prev_list])
    A_curr = np.array([c["area_km2"] for c in curr_list])
    # relative difference (prev x curr)
    rel_area = np.abs(A_prev[:,None] - A_curr[None,:]) / (A_prev[:,None] + 1e-6)

    # temp difference (mean_tb)
    T_prev = np.array([p["mean_tb"] for p in prev_list])
    T_curr = np.array([c["mean_tb"] for c in curr_list])
    dT = np.abs(T_prev[:,None] - T_curr[None,:])

    # Assemble cost (prev x curr)
    cost = w_dist * D.T + w_area * rel_area + w_temp * (dT / 10.0)  # scale temp term

    # Gating: too far -> invalidate
    invalid = D.T > max_dist_km
    # Optional: you could also gate on area ratio etc.

    # Inflate invalid pairs to large cost so they won't be chosen
    cost = np.where(invalid, 1e9, cost)
    return cost, invalid

# -------------------------
# Tracker
# -------------------------
class Track:
    def _init_(self, tid, obs, timestep, parent=None):
        self.id = tid
        self.history = []  # list of dicts per timestep
        self.misses = 0
        self.alive = True
        self.parent = parent
        self.add(obs, timestep)

    def add(self, obs, timestep):
        rec = {"t": timestep, **obs}
        self.history.append(rec)
        self.misses = 0

def track_sequence(hdf5_dir, mask_dir, max_misses=2, max_dist_km=500,
                   w_dist=1.0, w_area=0.3, w_temp=0.1, min_radius_km=100.0):
    tracks = []
    next_tid = 1
    prev_alive = []  # previous timestep active tracks' last obs

    rows = []

    for t, fname in enumerate(sorted(os.listdir(hdf5_dir))):
        if not fname.endswith(".h5"):
            continue

        fpath = os.path.join(hdf5_dir, fname)
        mpath = os.path.join(mask_dir, fname.replace(".h5", "_mask.npy"))
        if not os.path.exists(mpath):
            # advance time: increment misses on prev_alive
            for tr in tracks:
                if tr.alive and (len(tr.history)==0 or tr.history[-1]["t"] < t):
                    tr.misses += 1
                    if tr.misses > max_misses:
                        tr.alive = False
            continue

        with h5py.File(fpath, "r") as f:
            IMG = f["IMG_TIR1"][0]
            LUT = f["IMG_TIR1_TEMP"][:]
            X = f["X"][:]  # lon 1D
            Y = f["Y"][:]  # lat 1D

        IMG = np.clip(IMG, 0, LUT.shape[0]-1)
        T = LUT[IMG]
        mask = np.load(mpath).astype(bool)

        curr_clusters = extract_clusters_from_mask(mask, T, X, Y, min_radius_km=min_radius_km)

        # Build prev list (only alive tracks with last record at t-1)
        prev_list = []
        alive_ids = []
        for tr in tracks:
            if tr.alive and tr.history[-1]["t"] == t-1:
                prev_list.append(tr.history[-1])
                alive_ids.append(tr.id)

        # Match
        cost, invalid = build_cost(prev_list, curr_clusters, w_dist, w_area, w_temp, max_dist_km)
        assignments = []
        unassigned_prev = set(range(len(prev_list)))
        unassigned_curr = set(range(len(curr_clusters)))

        if len(prev_list) and len(curr_clusters):
            r, c = linear_sum_assignment(cost)  # r: prev idx, c: curr idx
            for i_prev, i_curr in zip(r, c):
                if cost[i_prev, i_curr] >= 1e9:  # gated out
                    continue
                assignments.append((i_prev, i_curr))
                unassigned_prev.discard(i_prev)
                unassigned_curr.discard(i_curr)

        # Update matched tracks
        for i_prev, i_curr in assignments:
            tid = alive_ids[i_prev]
            tr = next(tr for tr in tracks if tr.id == tid)
            obs = curr_clusters[i_curr].copy()
            tr.add(obs, t)

        # Handle unmatched previous (misses)
        for i_prev in unassigned_prev:
            tid = alive_ids[i_prev]
            tr = next(tr for tr in tracks if tr.id == tid)
            tr.misses += 1
            if tr.misses > max_misses:
                tr.alive = False

        # Handle unmatched current (new births)
        for i_curr in unassigned_curr:
            obs = curr_clusters[i_curr].copy()
            tr = Track(next_tid, obs, t)
            tracks.append(tr)
            next_tid += 1

        # OPTIONAL: split/merge detection
        # Count how many prev matched to each curr and vice versa
        if assignments:
            prev_to_curr = {}
            curr_to_prev = {}
            for ip, ic in assignments:
                prev_to_curr.setdefault(ip, []).append(ic)
                curr_to_prev.setdefault(ic, []).append(ip)
            # Split: one prev -> many curr
            for ip, currs in prev_to_curr.items():
                if len(currs) > 1:
                    # mark children with parent track id
                    parent_tid = alive_ids[ip]
                    for ic in currs:
                        # set parent on the track that just received obs ic
                        obs = curr_clusters[ic]
                        # find track with last obs == obs at t and matching centroid
                        # (simple heuristic; in production keep IDs by mapping)
                        pass  # keep as metadata if you like
            # Merge: many prev -> one curr (can tag metadata similarly)

        # Log rows
        for tr in tracks:
            if tr.alive and tr.history[-1]["t"] == t:
                last = tr.history[-1]
                rows.append({
                    "track_id": tr.id, "timestep": t,
                    "lat": last["lat"], "lon": last["lon"],
                    "area_km2": last["area_km2"],
                    "min_tb": last["min_tb"], "mean_tb": last["mean_tb"]
                })

    df = pd.DataFrame(rows).sort_values(["track_id", "timestep"]).reset_index(drop=True)
    return df,track